In [2]:
#### Analysis win rate ####
import os
import pandas as pd

# base_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/text-to-image/HPDv3/no_anime-hpdv3_all"
base_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3"

pickscore_df = pd.read_csv(os.path.join(base_dir, "pickscore/pickscore_score.csv"), dtype={"uid": str})
imagereward_df = pd.read_csv(os.path.join(base_dir, "imagereward/imagereward_score.csv"), dtype={"uid": str})
hpsv3_df = pd.read_csv(os.path.join(base_dir, "hpsv3/hpsv3_score.csv"), dtype={"uid": str})
deqa_df = pd.read_csv(os.path.join(base_dir, "deqa/deqa_score.csv"), dtype={"uid": str})
maagiqa_df = pd.read_csv(os.path.join(base_dir, "ma-agiqa/ma-agiqa_score.csv"), dtype={"uid": str})
aesthetic_df = pd.read_csv(os.path.join(base_dir, "aesthetic/aesthetic_score.csv"), dtype={"uid": str})
anime_df = pd.read_csv("/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3/qwen3_anime/qwen3_anime.csv", dtype={"uid": str})
valid_real_image_uids = anime_df[anime_df['anime'] == 'no']['uid'].tolist()

# colorfulness_filtered = colorfulness_df[colorfulness_df['uid'].isin(valid_real_image_uids)]
# avg_colorfulness = colorfulness_filtered['real_image_score'].mean()
# valid_colorfulness_uids = colorfulness_filtered[colorfulness_filtered['real_image_score'] < avg_colorfulness]['uid'].tolist()
pickscore_filtered = pickscore_df[pickscore_df['uid'].isin(valid_real_image_uids)]
pickscore_gap_threshold = 0.02
valid_pickscore_uids = pickscore_filtered[pickscore_filtered['real_image_score'] - pickscore_filtered['fake_image_score'] > pickscore_gap_threshold]['uid'].tolist()

# valid_uids = list(set(valid_colorfulness_uids) & set(valid_pickscore_uids))
# valid_uids = list(set(valid_brightness_uids) & set(valid_pickscore_uids))
valid_uids = valid_pickscore_uids
# valid_uids = valid_real_image_uids

pickscore_df = pickscore_df[pickscore_df['uid'].isin(valid_uids)]
imagereward_df = imagereward_df[imagereward_df['uid'].isin(valid_uids)]
hpsv3_df = hpsv3_df[hpsv3_df['uid'].isin(valid_uids)]
deqa_df = deqa_df[deqa_df['uid'].isin(valid_uids)]
maagiqa_df = maagiqa_df[maagiqa_df['uid'].isin(valid_uids)]
aesthetic_df = aesthetic_df[aesthetic_df['uid'].isin(valid_uids)]

# 计算每个reward model中满足 real_image_score > fake_image_score 的比例
# 注意：相同分数按0.5计算（平局）
reward_models = {
    'pickscore': pickscore_df,
    'imagereward': imagereward_df,
    'hpsv3': hpsv3_df,
    'deqa': deqa_df,
    'maagiqa': maagiqa_df,
    'aesthetic': aesthetic_df,
}

print(base_dir)
print("=" * 80)
print("Analysis: Percentage of UIDs where real_image_score > fake_image_score")
print("Note: Ties (real == fake) count as 0.5")
print("=" * 80)
print()

results = {}
for model_name, df in reward_models.items():
    total_count = len(df)
    win_count = len(df[df['real_image_score'] > df['fake_image_score']])
    tie_count = len(df[df['real_image_score'] == df['fake_image_score']])
    lose_count = len(df[df['real_image_score'] < df['fake_image_score']])
    score_diff_mean = (df['real_image_score'] - df['fake_image_score']).mean() if total_count > 0 else 0
    
    win_rate = ((win_count + tie_count * 0.5) / total_count * 100) if total_count > 0 else 0
    
    results[model_name] = {
        'total': total_count,
        'win': win_count,
        'tie': tie_count,
        'lose': lose_count,
        'win_rate': win_rate,
        'diff_mean': score_diff_mean
    }
    
    print(f"{model_name:15s}: Win={win_count:6d}, Tie={tie_count:6d}, Lose={lose_count:6d}")
    print(f"{'':15s}  Win Rate = {win_rate:6.2f}% ({(win_count + tie_count * 0.5):.1f} / {total_count})")
    print(f"{'':15s}  Mean(real - fake) = {score_diff_mean:.4f}")
    print()

print("=" * 80)
print("Summary Table")
print("=" * 80)
summary_df = pd.DataFrame(results).T
summary_df.columns = ['Total UIDs', 'Win', 'Tie', 'Lose', 'Win Rate (%)', 'Mean(real - fake)']
print(summary_df.to_string())

/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3
Analysis: Percentage of UIDs where real_image_score > fake_image_score
Note: Ties (real == fake) count as 0.5

pickscore      : Win= 10137, Tie=     0, Lose=     0
                 Win Rate = 100.00% (10137.0 / 10137)
                 Mean(real - fake) = 0.0438

imagereward    : Win=  6879, Tie=     0, Lose=  3258
                 Win Rate =  67.86% (6879.0 / 10137)
                 Mean(real - fake) = 0.3340

hpsv3          : Win=  8290, Tie=     0, Lose=  1847
                 Win Rate =  81.78% (8290.0 / 10137)
                 Mean(real - fake) = 2.3727

deqa           : Win=  7417, Tie=    48, Lose=  2672
                 Win Rate =  73.40% (7441.0 / 10137)
                 Mean(real - fake) = 0.2315

maagiqa        : Win=  8618, Tie=     0, Lose=  1519
                 Win Rate =  85.02% (8618.0 / 10137)
                 Mean(real - fake) = 0.2133

aesthetic      : Win=  6468, Tie=     0, Lose=  

In [ ]:
### write to csv ###
ext_list = [".png", ".PNG", ".jpg", ".jpeg", ".JPG", ".JPEG"]
real_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3/real"
fake_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3/fake"
all_information_df = pd.read_csv("/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/HPDv3/real_images_uid_prompt.csv", dtype={"uid": str})
total_information_list = []
for uid in valid_uids:
    for ext in ext_list:
        win_image_path = os.path.join(real_image_dir, f"{uid}{ext}")
        if os.path.exists(win_image_path):
            break
    if win_image_path is None:
        print(f"Warning: No valid image found for uid {uid}")
        
    for ext in ext_list:
        lose_image_path = os.path.join(fake_image_dir, f"{uid}{ext}")
        if os.path.exists(lose_image_path):
            break
    if lose_image_path is None:
        print(f"Warning: No valid image found for uid {uid}")
        continue
    
    prompt = all_information_df.loc[all_information_df['uid'] == uid, 'prompt'].values[0]
    total_information_list.append({
        'uid': uid,
        'prompt': prompt,
        'win_image_path': win_image_path,
        'lose_image_path': lose_image_path
    })
print(f"\nFinal valid samples: {len(total_information_list)}")
final_output_path = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3/no_anime_low_brightness-pickscore_002-hpdv3_all-uids.csv"
pd.DataFrame(total_information_list).to_csv(final_output_path, index=False)
print(f"Saved to: {final_output_path}")

In [ ]:
#### Link win image and lose image ####
import os
import pandas as pd

csv_file_name = "no_anime_low_brightness-pickscore_002-hpdv3_all-uids"
base_target_image_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3/visualization"
base_csv_dir = "/data_center/data2/dataset/chenwy/21164-data/dpo_dataset/u2net_next_inpainting/HPDv3"

ext_list = [".png", ".PNG", ".jpg", ".jpeg", ".JPG", ".JPEG"]
df = pd.read_csv(os.path.join(base_csv_dir, f"{csv_file_name}.csv"))
target_image_dir = os.path.join(base_target_image_dir, csv_file_name)
os.makedirs(os.path.join(target_image_dir, "real"), exist_ok=True)
os.makedirs(os.path.join(target_image_dir, "fake"), exist_ok=True)

for index, row in df.iterrows():
    uid = row['uid']
    source_win_image_path = row['win_image_path']
    source_lose_image_path = row['lose_image_path']

    win_image_name = os.path.basename(source_win_image_path)
    lose_image_name = os.path.basename(source_lose_image_path)

    target_win_image_path = os.path.join(target_image_dir, "real", win_image_name)
    target_lose_image_path = os.path.join(target_image_dir, "fake", lose_image_name)

    os.symlink(source_win_image_path, target_win_image_path)
    os.symlink(source_lose_image_path, target_lose_image_path)
